<a href="https://colab.research.google.com/github/GehadHelmy/github_SNA/blob/main/Project_SNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Libraries**

In [95]:
import pandas as pd
import numpy as np
import json
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

## **Dataset Understanding**

In [96]:
edges_df = pd.read_csv("musae_git_edges.csv")
target_df = pd.read_csv("musae_git_target.csv")
with open("musae_git_features.json", "r") as file:
    features_data = json.load(file)

In [97]:
print("Edges Shape:", edges_df.shape)
print("Target Shape:", target_df.shape)
print("Number of Feature Records:", len(features_data))

Edges Shape: (289003, 2)
Target Shape: (37700, 3)
Number of Feature Records: 37700


In [98]:
print("Edges:")
display(edges_df.head())
print("Target:")
display(target_df.head())

Edges:


,id_1,id_2
0,0,23977
1,1,34526
2,1,2370
3,1,14683
4,1,29982


Target:


,id,name,ml_target
0,0,Eiryyy,0
1,1,shawflying,0
2,2,JpMCarrilho,1
3,3,SuhwanCha,0
4,4,sunilangadi2,1


In [99]:
print(edges_df.columns)
print(target_df.columns)

Index(['id_1', 'id_2'], dtype='object')
Index(['id', 'name', 'ml_target'], dtype='object')


In [100]:
for i, (node_id, features) in enumerate(features_data.items()):
    print(node_id, features)
    if i == 4:
        break

0 [1574, 3773, 3571, 2672, 2478, 2534, 3129, 3077, 1171, 2045, 1539, 902, 1532, 2472, 1122, 2480, 3098, 2115, 1578]
1 [1193, 376, 73, 290, 3129, 1852, 3077, 1171, 1022, 2045, 536, 2040, 1533, 1532, 2472, 673, 798]
2 [1574, 3773, 925, 1728, 2815, 2963, 3077, 364, 1171, 536, 1867, 2472, 1122, 2532, 664, 28, 3311, 1768, 869]
3 [3964, 3773, 4003, 928, 1852, 3077, 364, 1022, 3763, 2045, 3859, 3771, 234, 664, 703]
4 [1929, 3773, 1793, 3511, 1290, 3129, 3077, 364, 1171, 1022, 2045, 536, 1119, 91, 1532, 3771, 1122, 703, 1001]


## **Data Preprocessing & Cleaning**

In [101]:
print("Edges Missing Values:")
print(edges_df.isnull().sum())
print("\nTarget Missing Values:")
print(target_df.isnull().sum())

Edges Missing Values:
id_1    0
id_2    0
dtype: int64

Target Missing Values:
id           0
name         0
ml_target    0
dtype: int64


In [102]:
edges_df["source"] = edges_df["id_1"].astype(int)

edges_df["target"] = edges_df["id_2"].astype(int)

edges_df["source_id"] = edges_df[["source", "target"]].min(axis=1)

edges_df["target_id"] = edges_df[["source", "target"]].max(axis=1)

print("Original Edges:", len(edges_df))
edges_df = edges_df.drop_duplicates(
    subset=["source_id", "target_id"])
edges_df = edges_df[["source_id", "target_id"]]
print("After Removing Duplicates:", len(edges_df))

Original Edges: 289003
After Removing Duplicates: 289003


In [103]:
self_loops = edges_df[
    edges_df["source_id"] == edges_df["target_id"]]
print("Number of Self Loops:", len(self_loops))

Number of Self Loops: 0


In [104]:
print("Duplicate Developer IDs:",target_df["id"].duplicated().sum())

Duplicate Developer IDs: 0


## **Graph Construction**

In [105]:
feature_nodes = set(map(int, features_data.keys()))
len(feature_nodes)

37700

In [106]:
G = nx.Graph()
G.add_nodes_from(feature_nodes)
G.add_edges_from(
    edges_df[["source_id", "target_id"]].itertuples(
        index=False,
        name=None
    ))
print("Number of Nodes:", G.number_of_nodes())
print("Number of Edges:", G.number_of_edges())

Number of Nodes: 37700
Number of Edges: 289003


### **Edge List**

In [107]:
edge_list = list(G.edges())
print(edge_list[:10])

[(0, 23977), (1, 34526), (1, 2370), (1, 14683), (1, 29982), (1, 21142), (1, 20363), (1, 23830), (1, 34035), (2, 3812)]


In [108]:
edge_list_df = pd.DataFrame(
    edge_list,
    columns=["Source", "Target"])
edge_list_df.to_csv("edge_list.csv", index=False)
edge_list_df.head()

,Source,Target
0,0,23977
1,1,34526
2,1,2370
3,1,14683
4,1,29982


### **Adjacency List**

In [109]:
adjacency_list = nx.to_dict_of_lists(G)
for node, neighbors in list(adjacency_list.items())[:5]:
    print(node, ":", neighbors)

0 : [23977]
1 : [34526, 2370, 14683, 29982, 21142, 20363, 23830, 34035]
2 : [3812]
3 : [4950, 18029, 3358, 34935, 5916]
4 : [2865, 9342]


In [110]:
with open("adjacency_list.json", "w") as file:
    json.dump(adjacency_list, file)

### **Adjacency Matrix**

In [111]:
from scipy.sparse import csr_matrix
A = nx.to_scipy_sparse_array(
    G,
    format="csr",
    dtype=np.int8)
print("Matrix Shape:", A.shape)
print("Number of Connections:", A.nnz)
print("Matrix Type:", type(A))

Matrix Shape: (37700, 37700)
Number of Connections: 578006
Matrix Type: <class 'scipy.sparse._csr.csr_array'>


In [113]:
import graphistry
from google.colab import userdata

username = userdata.get("GRAPHISTRY_USERNAME")
password = userdata.get("GRAPHISTRY_PASSWORD")

graphistry.register(
    api=3,
    username=username,
    password=password
)

g = graphistry.edges(
    edges_df,
    source="source_id",
    destination="target_id"
)

g2 = g.scene_settings(
    show_arrows=False,
    point_size=2.0,
    edge_curvature=0.0,
    point_opacity=1.0,
    edge_opacity=0.47
)

g2.plot()

ERROR:graphistry.utils.requests:HTTP 400 error - response content (JSON): {
  "non_field_errors": [
    "Unable to log in with provided credentials."
  ]
}
ERROR:graphistry.arrow_uploader:Error: <Response [400]>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/graphistry/arrow_uploader.py", line 331, in _finalize_login
    raise Exception(f"Login failed with HTTP {out.status_code}: {out.text}")
Exception: Login failed with HTTP 400: {"non_field_errors":["Unable to log in with provided credentials."]}


Exception: Login failed with HTTP 400: {"non_field_errors":["Unable to log in with provided credentials."]}

In [ ]:
graph_url = g2.plot(render=False)
print(graph_url)

## **Degree Analysis & Network Properties**

### **Node Degree**

In [ ]:
degree_dict = dict(G.degree())

degree_df = pd.DataFrame({
    "developer_id": list(degree_dict.keys()),
    "degree": list(degree_dict.values())
})

degree_df.head(10)

In [ ]:
degree_df['degree'].describe()

### **Average Degree**

In [ ]:
average_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()

print("Average Degree:", round(average_degree, 4))

### **Degree Distribution**

In [ ]:
degrees = [degree for node, degree in G.degree()]

degree_distribution = (
    pd.Series(degrees)
    .value_counts(normalize=True)
    .sort_index()
)

plt.figure(figsize=(10, 6))

plt.bar(
    degree_distribution.index,
    degree_distribution.values,
    color="steelblue",
    edgecolor="black"
)

plt.title("Degree Distribution of GitHub Developers")
plt.xlabel("Degree (k)")
plt.ylabel("P(k) = Nk / N")
plt.grid(axis="y", alpha=0.3)
plt.xlim(0, 100)

plt.show()

### **Top 10 Developers by Degree**

In [ ]:
top_10 = degree_df.sort_values(
    by="degree",
    ascending=False
).head(10)

top_10 = top_10.merge(
    target_df[["id", "name", "ml_target"]],
    left_on="developer_id",
    right_on="id",
    how="left"
)

top_10 = top_10[
    ["developer_id", "name", "degree", "ml_target"]
]

display(top_10)

In [ ]:
plt.figure(figsize=(11, 6))

sns.barplot(
    data=top_10,
    x="degree",
    y="name",
    color="steelblue"
)

plt.title("Top 10 GitHub Developers by Degree")
plt.xlabel("Number of Connections")
plt.ylabel("Developer")

plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

### **Clustering Coefficient**

In [ ]:
clustering_dict = nx.clustering(G)

clustering_df = pd.DataFrame({
    "developer_id": list(clustering_dict.keys()),
    "clustering_coefficient": list(clustering_dict.values())
})

display(clustering_df.head(10))

### **Highest 10 Clustering Coefficients**

In [ ]:
clustering_df = clustering_df.merge(
    target_df[["id", "name"]],
    left_on="developer_id",
    right_on="id",
    how="left"
)

clustering_df = clustering_df.drop(columns=["id"])


In [ ]:
top_10_clustering = clustering_df.sort_values(
    by="clustering_coefficient",
    ascending=False
).head(10)

display(top_10_clustering)

### **Lowest 10 Clustering Coefficients**

In [ ]:
bottom_10_clustering = clustering_df.sort_values(
    by="clustering_coefficient",
    ascending=True
).head(10)

display(bottom_10_clustering)

### **Average Clustering Coefficient**

In [ ]:
average_clustering = nx.average_clustering(G)

print("Average Clustering Coefficient:", round(average_clustering, 4))

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    clustering_df["clustering_coefficient"],
    bins=30,
    color="steelblue",
    edgecolor="black"
)

plt.title("Clustering Coefficient Distribution")
plt.xlabel("Clustering Coefficient")
plt.ylabel("Number of Developers")
plt.grid(axis="y", alpha=0.3)

plt.show()

## **Connectivity & Graph Structure**

### **Connected Components**

In [ ]:
connected_components = list(nx.connected_components(G))
num_components = len(connected_components)

print("Number of Connected Components:", num_components)

In [ ]:
component_sizes = [len(component) for component in connected_components]

component_sizes_df = pd.DataFrame({
    "component_id": range(1, num_components + 1),
    "number_of_developers": component_sizes
})

component_sizes_df

### **Special Graphs**

In [ ]:
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

is_complete = num_edges == (num_nodes * (num_nodes - 1)) / 2

degrees = [degree for node, degree in G.degree()]
is_regular = len(set(degrees)) == 1

is_tree = nx.is_tree(G)

is_bipartite = nx.is_bipartite(G)

special_graphs_df = pd.DataFrame({
    "Graph Property": [
        "Complete Graph",
        "Regular Graph",
        "Tree",
        "Bipartite Graph"
    ],
    "Result": [
        is_complete,
        is_regular,
        is_tree,
        is_bipartite
    ]
})

display(special_graphs_df)